# Subject-level SEG → CGM modeling

A reproducible walkthrough of the finalized retrospective workflow, with synthetic inputs and reusable figure code.

## 1. Scientific scope and architecture

Each subject is calibrated independently using their **earliest labeled event**. Later non-overlapping events are held out. SEG is the only sensor predictor; CGM provides calibration targets and manually labeled alignment landmarks.

**This is retrospective reconstruction, not prospective real-time prediction.** Full-event normalization, smoothing, and held-out CGM landmarks are used. Synthetic performance is a software demonstration, not evidence of clinical accuracy.

![CGM prediction workflow](CGM%20Prediction%20Modeling.png)

The supplied figure also illustrates case-level alternatives. This notebook implements the finalized **subject-level earliest-event calibration** arm only.

## 2. Environment and imports

Run this notebook from the CGM Prediction Modeling folder after installing `requirements.txt`. All code is local and independent of the original dashboards. No private patient files are loaded by default.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import matplotlib.pyplot as plt
import modeling
import plotting

ROOT = Path(modeling.__file__).resolve().parent
print("Local modeling and figure modules loaded.")


## 3. Configuration and feature equation

The finalized model is

$$\widehat{CGM}=b_0+b_1x+b_2x^2+b_3x^3+b_4\frac{dx}{dt}+b_5\phi.$$

- `x`: whole-event smoothed, min–max normalized SEG.
- `dx/dt`: native sensor-time slope per minute; zero across intervals outside 0.5–60 seconds and at each limb start.
- `φ`: signed, centered sigmoid of within-limb progress (positive direction on rise, negative on fall).
- Ridge standardization is fitted on calibration rows only, with an unpenalized intercept; returned coefficients are in the original feature basis.
- No coefficient or training sample is pooled between subjects.

In [ ]:
SMOOTHING_RATIO = 0.04
RIDGE_ALPHA = 1e-6
INPUT = ROOT / "data/synthetic_events.json"
OUTPUT = ROOT / "outputs/notebook_demo"
print({"smoothing": SMOOTHING_RATIO, "ridge_alpha": RIDGE_ALPHA,
       "minimum_calibration_samples": modeling.MIN_TRAIN_POINTS,
       "features": modeling.FEATURE_NAMES})


## 4. Load and inspect paired events

The portable JSON contains an `events` list. Each event has `subject_id`, `event_id`, `seg` and `cgm` observation arrays (`timestamp`, `value`), and three landmarks per signal (`trough1`, `peak`, `trough2`). CGM units are mmol/L; SEG values are sensor delta-current units before normalization.

The bundled data contain fictional identifiers and analytic curves. Inspect structure without printing all observation rows.

In [ ]:
events = modeling.load_events(INPUT)
print("Subjects:", len({e["subject_id"] for e in events}), "Events:", len(events))
print("Event keys:", list(events[0]))
print("Landmark keys:", {k: list(v) for k, v in events[0]["landmarks"].items()})
assert all(e["subject_id"].startswith("SYNTHETIC_") for e in events)


## 5. Inspect native-time inputs

Sensor and CGM peaks have different times and event durations. The following plots show the first event before warping. These traces are synthetic; their different units are shown on separate axes.

In [ ]:
event = events[0]
fig, axes = plt.subplots(2, 1, figsize=(9, 5))
for ax, signal, label in zip(axes, ["seg", "cgm"], ["SEG delta current", "CGM (mmol/L)"]):
    series = event[signal]
    times = [modeling.parse_time(p["timestamp"]) for p in series]
    minutes = [(t - times[0]).total_seconds() / 60 for t in times]
    ax.plot(minutes, [p["value"] for p in series], color="#007C91")
    for landmark, timestamp in event["landmarks"][signal].items():
        ax.axvline((modeling.parse_time(timestamp) - times[0]).total_seconds() / 60,
                   color="#777777", linestyle="--", linewidth=.7)
    ax.set(xlabel="Native minutes from first observation", ylabel=label)
fig.tight_layout()
plt.show()


## 6. Landmark warping and CGM-bin matching

1. Crop SEG to its labeled event, apply the original robust smoother, and normalize across that whole event.
2. Warp rising and falling limbs separately to their corresponding CGM intervals.
3. Compute `[1, x, x², x³, dx/dt, φ]` before averaging. Native-time slopes reset at the beginning of each limb.
4. Average expanded vectors in midpoint-bounded bins around the **original observed CGM samples**. Dense interpolated CGM values are not evaluation targets.
5. Deduplicate the shared peak sample; the falling-limb entry takes precedence, matching the finalized script.

Averaging `x²` differs from squaring the bin-average of `x`. Missing CGM bins produce no artificial target samples.

In [ ]:
prepared = modeling.prepare_event(event, smoothing=SMOOTHING_RATIO)
print("Native SEG observations:", len(event["seg"]))
print("Observed CGM samples retained:", len(prepared))
features = np.asarray([r["features"] for r in prepared])
assert features.shape[1] == 6 and np.isfinite(features).all()
assert len({r["mapped_dt"] for r in prepared}) == len(prepared)
print("Feature vector at first matched CGM sample:", features[0].round(5).tolist())


## 7. Chronological calibration and exclusions

Use the earliest validly identified event as calibration; do not select the best-performing event or silently substitute a later event if the earliest fails. Require at least **12 calibration CGM samples** and at least one later event with **3 retained samples**.

Validation timestamps at/before calibration end are removed. Duplicate timestamps across validation events are removed. Subjects that cannot meet these conditions are recorded as exclusions rather than pooled with another subject.

In [ ]:
subjects, sources, exclusions = modeling.prepare_subjects(events, SMOOTHING_RATIO)
for subject, split in subjects.items():
    train = [r for e in split["train"] for r in sources[e["id"]]]
    held_out = [r for e in split["validation"] for r in split["validation_rows"][e["id"]]]
    assert {r["mapped_dt"] for r in train}.isdisjoint(r["mapped_dt"] for r in held_out)
    assert all(r["mapped_dt"] > max(t["mapped_dt"] for t in train) for r in held_out)
    print(subject, {"calibration_samples": len(train), "held_out_samples": len(held_out),
                    "removed_overlap": split["overlap_removed"]})
print("Exclusions:", exclusions)
assert subjects, "No eligible subjects; inspect exclusions."


## 8. Fit one model per subject

The standardization means and population standard deviations are estimated only from that subject's calibration samples. Ridge α is fixed rather than tuned using held-out events. Coefficients are transformed back to the original six-feature basis for export.

In [ ]:
result = modeling.fit_subjects(subjects, sources, exclusions,
                              smoothing=SMOOTHING_RATIO, alpha=RIDGE_ALPHA)
result["input_sha256"] = hashlib.sha256(INPUT.read_bytes()).hexdigest()
result["synthetic"] = True
for subject, coefficients in result["coefficients"].items():
    print(subject, {name: round(value, 5) for name, value in coefficients.items()})


## 9. Evaluate calibration and held-out performance separately

MARD is the mean pointwise absolute relative error; RMSE is in mmol/L. R² may be negative and is undefined for constant targets. Subject summaries pool retained samples instead of averaging event MARD values. Pooled results below are sample-weighted across subjects and should be distinguished from subject-level summaries.

No MARD-based selection is applied in this notebook. Earlier selected-display figures are not used as unbiased performance estimates.

In [ ]:
for split in ["train", "validation"]:
    points = [p for p in result["points"] if p["split"] == split]
    print(split, modeling.metrics(points))
print("Per-subject metrics:")
for row in result["subject_metrics"]:
    print(row)


## 10. Prediction time-series figures

Each event gets its own facet, so lines never bridge separate events. Black is observed CGM; teal is calibration reconstruction; rust is held-out reconstruction. All retained points are included and y-limits are automatic. SVG and PNG are exported for editing and manuscript assembly.

In [ ]:
for subject in result["coefficients"]:
    fig = plotting.plot_subject_timeseries(result, subject, OUTPUT / "figures")
    plt.show()


## 11. Clarke error grid

Plot calibration and held-out samples separately. The implementation preserves the original figure's zone priority and **18.0 mg/dL per mmol/L** conversion. Zone classification occurs in mg/dL; axes are labeled in mmol/L. Shaded boundaries are drawn over the conventional 0–400 mg/dL window; limits expand if needed to keep all points visible.

The comparator is **observed CGM, not laboratory blood glucose**. Clarke zones do not establish clinical validation of the reconstruction. Negative/out-of-range predictions are not silently clipped or removed.

In [ ]:
clarke_summaries = {}
for split, title in [("train", "Calibration"), ("validation", "Held out")]:
    points = [p for p in result["points"] if p["split"] == split]
    fig, stats = plotting.plot_clarke(points, title, OUTPUT / "figures", f"clarke_{split}")
    clarke_summaries[split] = stats
    print(title, stats)
    plt.show()


## 12. Export predictions, coefficients, metrics, and exclusions

Outputs are separate from inputs. Re-running this notebook updates its configured output directory; choose a new `OUTPUT` to preserve a previous run. The command-line runner instead refuses to overwrite an existing run directory.

In [ ]:
modeling.export_results(result, OUTPUT / "data")
(OUTPUT / "data/clarke_summary.json").write_text(json.dumps(clarke_summaries, indent=2) + "\n")
print("Exported model_results.json, CSV tables, Clarke summaries, and PNG/SVG figures.")


## 13. Use your own annotated data

Two supported inputs:

- **Portable JSON:** replace `INPUT` with your own event file using the schema shown in `data/synthetic_events.json`.
- **Companion dashboard dataset:** use `modeling.load_dashboard_events(dataset_root)`, which reads `annotated_data/alignment_annotations.json`, referenced processed CSVs, and `raw_data/cgm/<participant>_cleaned.csv` from the three-folder layout. It uses SEG/CGM pairs and resolves SEG landmarks from saved indices where available. CGM files must contain numeric mmol/L readings.

For private input, remove the synthetic-only assertion in Section 4, set `result["synthetic"] = False` in Section 8, and set provenance to the actual input used. The CLI supports `--dashboard-root` directly. Keep private data outside this repository, use a separate output directory, and clear notebook outputs before sharing.

The adapter requires strictly increasing unique timestamps, finite observations, positive CGM, and observations covering all labeled landmarks. These input validation checks are explicit; it does not silently sort, impute, or extrapolate missing boundaries.

## 14. Reproducibility and implementation provenance

The extracted smoothing, feature binning, chronological split, and ridge routines originate from the finalized figure pipeline. `source_provenance.json` records source-code hashes only. No participant data, calibrated coefficients, original model-results JSON, or subject-specific display selections are bundled.

Run `python -m unittest discover -s tests -v` for mathematical and split checks. Use `python run_modeling.py --output outputs/new_run` for an equivalent non-interactive workflow. The architecture diagram explains the full original workflow; this notebook is intentionally limited to its finalized subject-level branch.